In [59]:
import random
random.seed(777)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score,precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import joblib
import json

In [72]:
df = pd.read_csv('../data/merged/merged_df_latest.csv')
df.shape

(909, 136)

In [75]:
# df.isna().sum().to_clipboard()
# df['subcategory_count'].value_counts()

In [3]:
#cols_predict represents the columns we have identified as dangerous symptoms 

df['imm_symp_loss_of_consciousness'].value_counts(), df['imm_symp_coma'].value_counts()

(imm_symp_loss_of_consciousness
 0.0    472
 1.0    428
 2.0      9
 Name: count, dtype: int64,
 imm_symp_coma
 0.0    897
 1.0     12
 Name: count, dtype: int64)

In [4]:
## as this will be our target variable, we will convert it to binary
df['imm_symp_loss_of_consciousness'] = np.where(df['imm_symp_loss_of_consciousness']>1,1, df['imm_symp_loss_of_consciousness'])
df['imm_symp_loss_of_consciousness'].value_counts()

imm_symp_loss_of_consciousness
0.0    472
1.0    437
Name: count, dtype: int64

In [5]:
num_cols = df.select_dtypes(include=['int', 'float']).columns
df[num_cols].isna().sum().to_clipboard()

In [6]:
cat_cols = df.select_dtypes(include=['object']).columns
print(cat_cols)

Index(['patient_id', 'date_of_birth', 'gender', 'patient_type', 'external_id',
       'first_tbi_date', 'last_tbi_date', 'first_tbi_from', 'first_tbi_desc',
       'immediate_symptoms_resulting', 'injury_from_new', 'subcategory',
       'severity', 'subcategory_count', 'severity_max', 'severity_min',
       'severity_mean', 'severity_std', 'severity_median', 'severity_var',
       'first_symptom_date', 'last_symptom_date', 'max_severity_date',
       'min_severity_date', 'day_of_max_severity', 'min_date_cognitive',
       'last_date_cognitive', 'min_date_emotional', 'last_date_emotional',
       'min_date_physical', 'last_date_physical', 'min_date_sleep',
       'last_date_sleep', 'min_date_speech', 'last_date_speech',
       'min_date_vision', 'last_date_vision', 'note1', 'note2', 'note3',
       'note1_post_date', 'note2_post_date', 'note3_post_date'],
      dtype='object')


In [7]:
# df['patient_sub_type'].value_counts()

In [8]:
df['imm_symp_coma'].sum(), df['imm_symp_loss_of_consciousness'].sum()

(np.float64(12.0), np.float64(437.0))

In [9]:
num_column_list = df[num_cols].columns.tolist()
print(num_column_list)

['Unnamed: 0', 'total_tbi', 'num_head_hit_location', 'headhit_Not_Sure', 'headhit_Neck', 'headhit_Top_Of_Head', 'headhit_Left_Side_Of_Head', 'headhit_Front_Of_Head', 'headhit_Back_Of_Head', 'headhit_All', 'headhit_Whiplash', 'headhit_Right_Side_Of_Head', 'imm_symp_light_sensitivity', 'imm_symp_headache', 'imm_symp_dazed_or_vacant_stare', 'imm_symp_dizziness', 'imm_symp_disorientation', 'imm_symp_nausea', 'imm_symp_confusion', 'imm_symp_coma', 'imm_symp_incoherent_speech', 'imm_symp_memory_loss', 'imm_symp_loss_of_consciousness', 'event_desc_car', 'event_desc_fall', 'event_desc_severe', 'injury_from_Accident', 'injury_from_Fall', 'injury_from_Collision', 'injury_from_Sports', 'injury_from_Assault', 'injury_from_Stroke', 'injury_from_Surgery', 'subcategory_count_cognitive', 'severity_max_cognitive', 'severity_min_cognitive', 'severity_mean_cognitive', 'severity_std_cognitive', 'severity_median_cognitive', 'severity_var_cognitive', 'first_date_cognitive', 'max_date_cognitive', 'subcategor

In [10]:
## Considering only numerical columns which doesn't have na values in it for model building.

num_cols_list = ['num_head_hit_location', 'headhit_Not_Sure', 'headhit_Neck', 'headhit_Top_Of_Head', 
                 'headhit_Left_Side_Of_Head', 'headhit_Front_Of_Head', 'headhit_Back_Of_Head', 
                 'headhit_All', 'headhit_Whiplash', 'headhit_Right_Side_Of_Head',
# Immediate Symptoms
'imm_symp_light_sensitivity', 'imm_symp_headache', 'imm_symp_dazed_or_vacant_stare', 'imm_symp_dizziness',
 'imm_symp_disorientation', 'imm_symp_nausea', 'imm_symp_confusion', 'imm_symp_coma', 'imm_symp_incoherent_speech',
   'imm_symp_memory_loss', 'imm_symp_loss_of_consciousness',
# Event Descriptions
'event_desc_car', 'event_desc_fall', 'event_desc_severe'
# Injury from
, 'injury_from_Accident', 'injury_from_Fall', 'injury_from_Collision', 'injury_from_Sports',
'injury_from_Assault', 'injury_from_Stroke', 'injury_from_Surgery', 'age_tbi'
]


cat_cols_list = ['patient_id', 'gender']

In [11]:
# applying minmax scaling to patients age column
scaler = StandardScaler()
df['age_tbi'] = scaler.fit_transform(df[['age_tbi']])

# df['age_tbi'] = (df['age_tbi'] - df['age_tbi'].min()) / (df['age_tbi'].max() - df['age_tbi'].min())

In [12]:
df['age_tbi'].describe()

count    9.080000e+02
mean     1.721579e-16
std      1.000551e+00
min     -4.864799e+00
25%     -7.473261e-01
50%      1.305433e-03
75%      7.499369e-01
max      4.430709e+00
Name: age_tbi, dtype: float64

In [13]:
joblib.dump(scaler, '../artifacts/scaler_age_tbi.pkl')

['../artifacts/scaler_age_tbi.pkl']

In [14]:
final_df = df[cat_cols_list + num_cols_list].copy(deep=True)
final_df.dropna(inplace=True)
final_df.reset_index(drop=True, inplace=True)
final_df.shape

(908, 34)

In [15]:
# final_df.isna().sum()

In [16]:
## encoding for the gender column
final_df = pd.get_dummies(final_df, columns=['gender'], prefix='gender', drop_first=True)
final_df.shape

(908, 35)

In [17]:
final_df.columns

Index(['patient_id', 'num_head_hit_location', 'headhit_Not_Sure',
       'headhit_Neck', 'headhit_Top_Of_Head', 'headhit_Left_Side_Of_Head',
       'headhit_Front_Of_Head', 'headhit_Back_Of_Head', 'headhit_All',
       'headhit_Whiplash', 'headhit_Right_Side_Of_Head',
       'imm_symp_light_sensitivity', 'imm_symp_headache',
       'imm_symp_dazed_or_vacant_stare', 'imm_symp_dizziness',
       'imm_symp_disorientation', 'imm_symp_nausea', 'imm_symp_confusion',
       'imm_symp_coma', 'imm_symp_incoherent_speech', 'imm_symp_memory_loss',
       'imm_symp_loss_of_consciousness', 'event_desc_car', 'event_desc_fall',
       'event_desc_severe', 'injury_from_Accident', 'injury_from_Fall',
       'injury_from_Collision', 'injury_from_Sports', 'injury_from_Assault',
       'injury_from_Stroke', 'injury_from_Surgery', 'age_tbi', 'gender_male',
       'gender_other'],
      dtype='object')

### Model Building

In [18]:
final_df['imm_symp_loss_of_consciousness'].value_counts()

imm_symp_loss_of_consciousness
0.0    472
1.0    436
Name: count, dtype: int64

In [22]:
cols_predict = ['imm_symp_loss_of_consciousness', 'imm_symp_coma']

models = {
    "LogisticRegression": LogisticRegression(max_iter=3000, class_weight='balanced'),
    "RandomForest": RandomForestClassifier(class_weight='balanced', random_state=123),
    "XGBoost": xgb.XGBClassifier(eval_metric='mlogloss', use_label_encoder=False)
}

param_grids = {
    "LogisticRegression": {"class_weight": ['balanced'], 'C': [0.1, 1.0, 10.0]},
    "RandomForest": {"n_estimators": [100, 300, 500], "max_depth": [4, 8]},
    "XGBoost": {"max_depth": [4, 8], "subsample": [0.7, 1.0], "eta": [0.1, 0.05]}
}


In [23]:
# STEP 3: Run models and collect results
model_results = []
feature_importances = []
prediction_outputs = final_df[["patient_id"]].copy()  # for probability mapping

for col in cols_predict:
    df_ = final_df.drop(columns=[c for c in cols_predict if c != col])
    X = df_.drop(columns=[col, "patient_id"])
    y = df_[col]
    patient_ids = df_["patient_id"]

    train_X, test_X, train_y, test_y = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=123
    )

    for model_name, model_instance in models.items():
        if param_grids[model_name]:
            grid = GridSearchCV(model_instance, param_grids[model_name], cv=3)
            grid.fit(train_X, train_y)
            model = grid.best_estimator_
        else:
            model = model_instance.fit(train_X, train_y)

        y_pred = model.predict(test_X)
        y_proba = model.predict_proba(X)[:, 1]  # probability of class 1

        # Map predictions back to patient_id
        prediction_outputs[f"predict_{col}_{model_name}"] = y_proba

        # Handle classification report safely
        report = classification_report(test_y, y_pred, output_dict=True)
        model_results.append({
            "Symptom": col,
            "Model": model_name,
            "Accuracy": accuracy_score(test_y, y_pred),
            "Precision": precision_score(test_y, y_pred, average='binary', zero_division=0),
            "Recall": recall_score(test_y, y_pred, average='binary', zero_division=0),
            "F1-Score": f1_score(test_y, y_pred, average='binary', zero_division=0)
        })

        # Extract feature importances
        if hasattr(model, 'coef_'):  # Logistic Regression
            for feat, imp in zip(X.columns, model.coef_[0]):
                feature_importances.append({
                    "Symptom": col,
                    "Model": model_name,
                    "Feature": feat,
                    "Importance": abs(imp)
                })
        elif hasattr(model, 'feature_importances_'):  # RF, XGBoost
            for feat, imp in zip(X.columns, model.feature_importances_):
                feature_importances.append({
                    "Symptom": col,
                    "Model": model_name,
                    "Feature": feat,
                    "Importance": imp
                })

# STEP 4: Compile results
results_df = pd.DataFrame(model_results)
feature_df = pd.DataFrame(feature_importances)

# STEP 5: Merge final_df with prediction outputs
final_with_preds = final_df.merge(prediction_outputs, on="patient_id", how="left")


d:\workspace\git_projects\Power-of-Patients-Capstone\.venv\lib\site-packages\xgboost\training.py:183: UserWarning: [00:43:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\workspace\git_projects\Power-of-Patients-Capstone\.venv\lib\site-packages\xgboost\training.py:183: UserWarning: [00:43:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\workspace\git_projects\Power-of-Patients-Capstone\.venv\lib\site-packages\xgboost\training.py:183: UserWarning: [00:43:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\workspace\git_projects\Power-of-Patients-Capstone\.venv\lib\site-packages\xgboost\training.py:183: UserWarning: [00:44:00] W

In [24]:
# Show outputs
print("\n=== Model Evaluation Results ===")
results_df


=== Model Evaluation Results ===


,Symptom,Model,Accuracy,Precision,Recall,F1-Score
0,imm_symp_loss_of_consciousness,LogisticRegression,0.692308,0.657718,0.748092,0.700000
1,imm_symp_loss_of_consciousness,RandomForest,0.688645,0.676923,0.671756,0.674330
2,imm_symp_loss_of_consciousness,XGBoost,0.699634,0.692913,0.671756,0.682171
3,imm_symp_coma,LogisticRegression,0.897436,0.038462,0.250000,0.066667
4,imm_symp_coma,RandomForest,0.985348,0.000000,0.000000,0.000000
5,imm_symp_coma,XGBoost,0.985348,0.000000,0.000000,0.000000


In [25]:
print("\n=== Feature Importances ===")
feature_df.sort_values(by=["Symptom", "Model", "Importance"], ascending=False)



=== Feature Importances ===


,Symptom,Model,Feature,Importance
82,imm_symp_loss_of_consciousness,XGBoost,imm_symp_memory_loss,0.182169
81,imm_symp_loss_of_consciousness,XGBoost,imm_symp_incoherent_speech,0.068214
69,imm_symp_loss_of_consciousness,XGBoost,headhit_Front_Of_Head,0.050408
73,imm_symp_loss_of_consciousness,XGBoost,headhit_Right_Side_Of_Head,0.039513
89,imm_symp_loss_of_consciousness,XGBoost,injury_from_Sports,0.036182
...,...,...,...,...
100,imm_symp_coma,LogisticRegression,headhit_Left_Side_Of_Head,0.565554
108,imm_symp_coma,LogisticRegression,imm_symp_dazed_or_vacant_stare,0.488423
97,imm_symp_coma,LogisticRegression,headhit_Not_Sure,0.336441
96,imm_symp_coma,LogisticRegression,num_head_hit_location,0.230407


In [26]:
feature_df.sort_values(by=["Symptom", "Model", "Importance"], ascending=False).to_clipboard(index=False)

## Let's finalize the logistic regression model 

In [19]:
final_df['imm_symp_loss_of_consciousness'].value_counts()

imm_symp_loss_of_consciousness
0.0    472
1.0    436
Name: count, dtype: int64

In [20]:
# STEP 1: Setup
cols_predict = ['imm_symp_loss_of_consciousness', 'imm_symp_coma']

model_results = []
feature_importances = []
prediction_outputs = final_df[["patient_id"]].copy()

logistic_model = LogisticRegression(max_iter=3000)
param_grid = {"class_weight": ['balanced'], 'C': [0.1, 1.0, 10.0]}

# STEP 2: Loop through each binary target
for col in cols_predict:
    df_ = final_df.drop(columns=[c for c in cols_predict if c != col])
    X = df_.drop(columns=[col, "patient_id"])
    y = df_[col].astype(int)  # ensure binary int
    patient_ids = df_["patient_id"]

    if y.nunique() != 2:
        print(f"Skipping {col} because it is not binary.")
        continue

    train_X, test_X, train_y, test_y = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=123
    )

    grid = GridSearchCV(logistic_model, param_grid, cv=3)
    grid.fit(train_X, train_y)
    best_model = grid.best_estimator_


    model_path = f"../artifacts/logreg_model_{col}.pkl"
    joblib.dump(best_model, model_path)

    # Save the feature order used for training
    feature_order_path = f"../artifacts/features_{col}.json"
    with open(feature_order_path, "w") as f:
        json.dump(list(X.columns), f)

    # Save model metadata (best params)
    params_path = f"../artifacts/model_params_{col}.json"
    with open(params_path, "w") as f:
        json.dump(grid.best_params_, f)

    # Predict
    y_train_pred = best_model.predict(train_X)
    y_test_pred = best_model.predict(test_X)
    y_proba = best_model.predict_proba(df_[X.columns])[:, 1]  # Full data for probability

    # Store probabilities
    prediction_outputs[f"predict_{col}_LogisticRegression"] = y_proba

    # Store both train and test metrics
    for split_name, true_y, pred_y in [
        ("Train", train_y, y_train_pred),
        ("Test", test_y, y_test_pred)
    ]:
        model_results.append({
            "Symptom": col,
            "Model": "LogisticRegression",
            "Dataset": split_name,
            "Accuracy": accuracy_score(true_y, pred_y),
            "Precision": precision_score(true_y, pred_y, average='binary', zero_division=0),
            "Recall": recall_score(true_y, pred_y, average='binary', zero_division=0),
            "F1-Score": f1_score(true_y, pred_y, average='binary', zero_division=0)
        })

    # Feature importances
    for feat, imp in zip(X.columns, best_model.coef_[0]):
        feature_importances.append({
            "Symptom": col,
            "Model": "LogisticRegression",
            "Feature": feat,
            "Importance": abs(imp)
        })

# STEP 3: Compile results
results_df = pd.DataFrame(model_results)
feature_df = pd.DataFrame(feature_importances)

In [21]:
results_df

,Symptom,Model,Dataset,Accuracy,Precision,Recall,F1-Score
0,imm_symp_loss_of_consciousness,LogisticRegression,Train,0.741732,0.735786,0.721311,0.728477
1,imm_symp_loss_of_consciousness,LogisticRegression,Test,0.692308,0.657718,0.748092,0.700000
2,imm_symp_coma,LogisticRegression,Train,0.922835,0.140351,1.000000,0.246154
3,imm_symp_coma,LogisticRegression,Test,0.897436,0.038462,0.250000,0.066667


In [22]:
feature_df.sort_values(by=["Symptom", "Model", "Importance"], ascending=False).to_clipboard(index=False)

In [23]:
prediction_outputs.to_clipboard(index=False)

In [24]:
final_with_preds = final_df.merge(prediction_outputs, on="patient_id", how="left")


In [ ]:
final_with_preds[['patient_id', 'imm_symp_loss_of_consciousness', 'imm_symp_coma',
                 'predict_imm_symp_loss_of_consciousness_LogisticRegression',
                 'predict_imm_symp_coma_LogisticRegression']].to_clipboard(index=False)

In [26]:
final_with_preds.head(2)

,patient_id,num_head_hit_location,headhit_Not_Sure,headhit_Neck,headhit_Top_Of_Head,headhit_Left_Side_Of_Head,headhit_Front_Of_Head,headhit_Back_Of_Head,headhit_All,headhit_Whiplash,...,injury_from_Collision,injury_from_Sports,injury_from_Assault,injury_from_Stroke,injury_from_Surgery,age_tbi,gender_male,gender_other,predict_imm_symp_loss_of_consciousness_LogisticRegression,predict_imm_symp_coma_LogisticRegression
0,5c96ba1a-8b2d-49bc-8e8e-b07761948286,6,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.500393,False,False,0.882477,0.000095
1,eda39327-b38f-41de-a46a-8782787369b7,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,-0.622554,True,False,0.780085,0.223576


## IISS


$ IISS = \beta_{s} * \sum_{i} w_i s_i + \beta_{h} * \sum_{j} w_j h_j + \beta_{e} * \sum_{k} w_k e_k + \beta_{d} * \sum_{l} d_l $

In [38]:
#Import the weights
weights = pd.read_csv("../data/raw/weights.csv")
weights.shape

(32, 2)

In [39]:
#grab imm_symptoms weight
imm_symp_weights = weights[weights["Column"].str.contains("imm_symp")]
symp_list = imm_symp_weights.Column.to_list()
#grab headhit_location weights
headhit_weights = weights[weights["Column"].str.contains("headhit")]
headhit_list = headhit_weights.Column.to_list()
#grab injury from weights
injury_from_weights = weights[weights["Column"].str.contains("injury_from")]
injury_from_list = injury_from_weights.Column.to_list()
#grab demograc=phic weight (we only have 1 so no need to convert to list)
demographic_weight = weights[weights["Column"].str.contains("demographic")]

In [40]:
final_with_preds.head()

,patient_id,num_head_hit_location,headhit_Not_Sure,headhit_Neck,headhit_Top_Of_Head,headhit_Left_Side_Of_Head,headhit_Front_Of_Head,headhit_Back_Of_Head,headhit_All,headhit_Whiplash,...,injury_from_Collision,injury_from_Sports,injury_from_Assault,injury_from_Stroke,injury_from_Surgery,age_tbi,gender_male,gender_other,predict_imm_symp_loss_of_consciousness,predict_imm_symp_coma
0,5c96ba1a-8b2d-49bc-8e8e-b07761948286,6,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.500393,False,False,0.882477,0.000095
1,eda39327-b38f-41de-a46a-8782787369b7,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,-0.622554,True,False,0.780085,0.223576
2,6a7f7ce9-f63e-4651-b941-a25ce116de74,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.685726,False,False,0.569828,0.000001
3,096d402a-5fcd-41dd-b59b-b3089cf06742,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.126077,True,False,0.285905,0.001289
4,a419b210-7615-40ac-ad05-a9e0a83046af,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.375621,False,False,0.475002,0.000004


In [58]:
final_with_preds.columns

Index(['patient_id', 'num_head_hit_location', 'headhit_not_sure',
       'headhit_neck', 'headhit_top_of_head', 'headhit_left_side_of_head',
       'headhit_front_of_head', 'headhit_back_of_head', 'headhit_all',
       'headhit_whiplash', 'headhit_right_side_of_head',
       'imm_symp_light_sensitivity', 'imm_symp_headache',
       'imm_symp_dazed_or_vacant_stare', 'imm_symp_dizziness',
       'imm_symp_disorientation', 'imm_symp_nausea', 'imm_symp_confusion',
       'imm_symp_coma', 'imm_symp_incoherent_speech', 'imm_symp_memory_loss',
       'imm_symp_loss_of_consciousness', 'event_desc_car', 'event_desc_fall',
       'event_desc_severe', 'injury_from_accident', 'injury_from_fall',
       'injury_from_collision', 'injury_from_sports', 'injury_from_assault',
       'injury_from_stroke', 'injury_from_surgery', 'age_tbi', 'gender_male',
       'gender_other', 'predict_imm_symp_loss_of_consciousness',
       'predict_imm_symp_coma', 'IISS'],
      dtype='object')

In [41]:
patients = final_with_preds
patients.shape

(908, 37)

In [42]:
patients.rename(columns={'predict_imm_symp_loss_of_consciousness_LogisticRegression': 'predict_imm_symp_loss_of_consciousness',
                         'predict_imm_symp_coma_LogisticRegression': 'predict_imm_symp_coma'}, inplace=True)

In [43]:
symp_list = [symp.lower() for symp in symp_list]
symp_list

['imm_symp_memory_loss',
 'imm_symp_loss_of_consciousness',
 'imm_symp_confusion',
 'imm_symp_dazed_or_vacant_stare',
 'imm_symp_disorientation',
 'imm_symp_incoherent_speech',
 'imm_symp_coma',
 'imm_symp']

In [44]:
#IMMEDIATE SYMPTOM SCORES

#initialize with a score of 0
imm_symp_score = 0
for i in range(len(symp_list[:-1])):
    
    if 'coma' in symp_list[i]: #LOOK AT COMA PREDICTIONS
        coma = []
        for index, r in patients.iterrows(): #APPEND WIEGHTS
            prob_column = 'predict_' + symp_list[i]
            
            #append scores by multipliyng with coma weights (using iloc[i]
            if r[symp_list[i]] == 1: #PATIENT HAS COMA
                coma.append(r[symp_list[i]]*imm_symp_weights.iloc[i][1])
            elif r[prob_column] > .5: #THIS IS THE THRESHOLD FOR NON-COMA
                coma.append(r[prob_column]*imm_symp_weights.iloc[i][1])          
            else:
                coma.append(0)
        #ADD COMA SCORE TOTAL SCORE       
        imm_symp_score.add(pd.Series(coma))
        
    #SAME AS BEFORE BUT NOW FOR LOSS OF CONSCIOUSNESS
    elif 'loss_of_consciousness' in symp_list[i]:
        print("LOSS")
        con = []
        for index, r in patients.iterrows():
            prob_column = 'predict_' + symp_list[i]
            if r[symp_list[i]] == 1:
                con.append(r[symp_list[i]]*imm_symp_weights.iloc[i][1])
            elif r[prob_column] > .5:
                # print(r[prob_column])
                con.append(r[prob_column]*imm_symp_weights.iloc[i][1])          
            else:
                con.append(0)
        imm_symp_score.add(pd.Series(con)) 
    else: #WE DO THIS FOR ALL OTHER IMM_SYMP'=
        #multiply whether a patient has the symptom times the symptom weight
        #and add it to the score
        imm_symp_score += patients[symp_list[i]]*imm_symp_weights.iloc[i][1]
        
#multiply by beta (beta is always the last row of its weights list)
imm_symp_score *= imm_symp_weights.iloc[-1][1]

LOSS


C:\Users\rohit\AppData\Local\Temp\ipykernel_10056\478199748.py:39: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  imm_symp_score += patients[symp_list[i]]*imm_symp_weights.iloc[i][1]
C:\Users\rohit\AppData\Local\Temp\ipykernel_10056\478199748.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  con.append(r[symp_list[i]]*imm_symp_weights.iloc[i][1])
C:\Users\rohit\AppData\Local\Temp\ipykernel_10056\478199748.py:32: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by posi

In [45]:
patients.head()

,patient_id,num_head_hit_location,headhit_Not_Sure,headhit_Neck,headhit_Top_Of_Head,headhit_Left_Side_Of_Head,headhit_Front_Of_Head,headhit_Back_Of_Head,headhit_All,headhit_Whiplash,...,injury_from_Collision,injury_from_Sports,injury_from_Assault,injury_from_Stroke,injury_from_Surgery,age_tbi,gender_male,gender_other,predict_imm_symp_loss_of_consciousness,predict_imm_symp_coma
0,5c96ba1a-8b2d-49bc-8e8e-b07761948286,6,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.500393,False,False,0.882477,0.000095
1,eda39327-b38f-41de-a46a-8782787369b7,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,-0.622554,True,False,0.780085,0.223576
2,6a7f7ce9-f63e-4651-b941-a25ce116de74,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.685726,False,False,0.569828,0.000001
3,096d402a-5fcd-41dd-b59b-b3089cf06742,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.126077,True,False,0.285905,0.001289
4,a419b210-7615-40ac-ad05-a9e0a83046af,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.375621,False,False,0.475002,0.000004


In [51]:
headhit_list = [hit.lower() for hit in headhit_list]
headhit_list

['headhit_front_of_head',
 'headhit_back_of_head',
 'headhit_left_side_of_head',
 'headhit_right_side_of_head',
 'headhit_top_of_head',
 'headhit_neck',
 'headhit_back_of_head',
 'headhit']

In [47]:
injury_from_list = [injury.lower() for injury in injury_from_list]
injury_from_list

['injury_from_accident',
 'injury_from_fall',
 'injury_from_collision',
 'injury_from_assault',
 'injury_from_sports',
 'injury_from_stroke',
 'injury_from_surgery',
 'injury_from']

In [48]:
patients.columns = [col.strip().lower() for col in patients.columns]

In [49]:
patients.columns

Index(['patient_id', 'num_head_hit_location', 'headhit_not_sure',
       'headhit_neck', 'headhit_top_of_head', 'headhit_left_side_of_head',
       'headhit_front_of_head', 'headhit_back_of_head', 'headhit_all',
       'headhit_whiplash', 'headhit_right_side_of_head',
       'imm_symp_light_sensitivity', 'imm_symp_headache',
       'imm_symp_dazed_or_vacant_stare', 'imm_symp_dizziness',
       'imm_symp_disorientation', 'imm_symp_nausea', 'imm_symp_confusion',
       'imm_symp_coma', 'imm_symp_incoherent_speech', 'imm_symp_memory_loss',
       'imm_symp_loss_of_consciousness', 'event_desc_car', 'event_desc_fall',
       'event_desc_severe', 'injury_from_accident', 'injury_from_fall',
       'injury_from_collision', 'injury_from_sports', 'injury_from_assault',
       'injury_from_stroke', 'injury_from_surgery', 'age_tbi', 'gender_male',
       'gender_other', 'predict_imm_symp_loss_of_consciousness',
       'predict_imm_symp_coma'],
      dtype='object')

In [52]:
headhit_score = 0
#get headhit scores
for i in range(len(headhit_list[:-2])):
    headhit_score += patients[headhit_list[i]]*headhit_weights.iloc[i][1] 
#multiply by beta (beta is always the last row of its wights list)
headhit_score *= headhit_weights.iloc[-1][1]

injury_from_score = 0
#get injury_from scores
for i in range(len(injury_from_list[:-1])):
    injury_from_score += patients[injury_from_list[i]]*injury_from_weights.iloc[i][1] 
#multiply by beta
injury_from_score *= injury_from_weights.iloc[-1][1]

#demographic score (multiply by age, then sum(gender = 1 so just add the demographic weight)
demographic_score = patients["age_tbi"]*demographic_weight.iloc[0][1] + demographic_weight.iloc[0][1]

C:\Users\rohit\AppData\Local\Temp\ipykernel_10056\4004217539.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  headhit_score += patients[headhit_list[i]]*headhit_weights.iloc[i][1]
C:\Users\rohit\AppData\Local\Temp\ipykernel_10056\4004217539.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  headhit_score *= headhit_weights.iloc[-1][1]
C:\Users\rohit\AppData\Local\Temp\ipykernel_10056\4004217539.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use

In [53]:
#check max and min scores
max(imm_symp_score + headhit_score + injury_from_score + demographic_score)
#should also normalize based on this into a more useful range (i.e. 0-100)

0.861561278364237

In [54]:
#ADD UP ALL INDIVIDUAL SCORES FOR IISS AND ADD IT TO DF
patients["IISS"] = imm_symp_score + headhit_score + injury_from_score + demographic_score

In [55]:
patients['IISS'].describe()

count    908.000000
mean       0.257187
std        0.147800
min       -0.443924
25%        0.150619
50%        0.258250
75%        0.351817
max        0.861561
Name: IISS, dtype: float64

In [74]:
patients.to_csv("../data/IISS_score.csv", index=False)

In [56]:
patients.head(2)

,patient_id,num_head_hit_location,headhit_not_sure,headhit_neck,headhit_top_of_head,headhit_left_side_of_head,headhit_front_of_head,headhit_back_of_head,headhit_all,headhit_whiplash,...,injury_from_sports,injury_from_assault,injury_from_stroke,injury_from_surgery,age_tbi,gender_male,gender_other,predict_imm_symp_loss_of_consciousness,predict_imm_symp_coma,IISS
0,5c96ba1a-8b2d-49bc-8e8e-b07761948286,6,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.500393,False,False,0.882477,0.000095,0.396451
1,eda39327-b38f-41de-a46a-8782787369b7,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,-0.622554,True,False,0.780085,0.223576,0.218268


In [57]:
patients.head(2).to_clipboard(index=False)